# QLoRA Analytics Agent — Local RTX 5090 Training

Primary target: **RTX 5090 (32GB VRAM)** with Qwen2.5-3B-Instruct QLoRA.
Batch size auto-adapts to 32GB (bs=4, grad_accum=4) with gradient checkpointing + packing.

Run this notebook from the repository root (`qlora-analytics-agent/`).

## 0. Environment check

In [ ]:
import sys, torch
sys.path.insert(0, '.')
from src.train import detect_hardware, auto_batch_size
hw = detect_hardware(); print(hw)
print('auto batch/grad-accum:', auto_batch_size(hw['vram_gb']))

## 1. Generate + preprocess synthetic data (deterministic, seed=42)

In [ ]:
!python scripts/01_generate_synthetic_db.py
!python scripts/02_generate_instruction_data.py
!python scripts/03_preprocess.py

## 2. Train QLoRA (r=16 default) + rank ablations

In [ ]:
!python scripts/04_train_qlora.py --config configs/train_qlora_3b.yaml --lora_r 16
!python scripts/04_train_qlora.py --lora_r 8
!python scripts/04_train_qlora.py --lora_r 32

## 3. Evaluate all conditions + plot

In [ ]:
!python scripts/05_evaluate.py
!python scripts/06_plot_results.py

## 4. Inspect results inline

In [ ]:
import json
from IPython.display import Image, display
print(json.dumps(json.load(open('results/metrics.json')), indent=2))
for fig in ['routing_accuracy', 'sql_exec_accuracy', 'safety_json_metrics', 'training_loss', 'resource_comparison']:
    p = f'figures/{fig}.png'
    import os
    if os.path.exists(p):
        display(Image(p))

## 5. Try the trained agent on a single question

In [ ]:
from src.inference import HFPredictor
from src.preprocess import SYSTEM_PROMPT, build_user_prompt
from src.schema import load_schema, load_data_dictionary, render_business_rules

schema = load_schema(); rules = render_business_rules(load_data_dictionary())
rec = {'question': 'What is the gross margin percentage by category?'}
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': build_user_prompt(rec, schema, rules)},
]
pred = HFPredictor(adapter_dir='results/adapters/qlora_r16')
contract, raw, latency = pred.predict(messages)
print('latency_ms=', round(latency, 1)); print(contract)